# Module 2, Notebook 2: Monte Carlo Control

## Learning Objectives
By completing this notebook, you will:
1. Implement on-policy MC control with epsilon-soft policies from scratch
2. Train an agent on a custom GridWorld without any environment model
3. Visualize the learned policy as an arrow grid and the Q function as a heatmap
4. Understand the exploration-exploitation trade-off through the epsilon parameter
5. Compare learned policies across different epsilon values to see the impact of exploration

## Prerequisites
- Module 2, Notebook 1: MC prediction, first-visit returns, episode sampling
- Module 1, Notebook 2: policy improvement (greedy policy extraction)
- Module 0, Notebook 1: epsilon-greedy action selection

## Estimated Time: 15 minutes

---

**Key Insight:** MC control is generalized policy iteration without a model. Instead of computing $Q^\pi$ analytically (impossible without $P$ and $R$), we estimate it from sampled returns. The catch: we must explore enough to estimate Q for all state-action pairs — this is what epsilon-soft policies handle.

## 1. Setup

In [ ]:
# Purpose: Imports and reproducibility
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import TwoSlopeNorm

np.random.seed(42)
print("Ready. NumPy version:", np.__version__)

## 2. From Prediction to Control

MC **prediction** estimates $V^\pi$ or $Q^\pi$ for a fixed policy $\pi$.  
MC **control** finds the optimal policy $\pi^*$ by alternating:

1. **Estimation**: estimate $Q(s,a)$ from sampled returns under current policy
2. **Improvement**: update policy to be greedy (or near-greedy) w.r.t. estimated Q

**Why Q and not V?** Without a model, we cannot compute $\arg\max_a \sum_{s'} P(s'|s,a)[R + V(s')]$ — it requires $P$. But $\arg\max_a Q(s,a)$ needs only the Q table.

**Why epsilon-soft?** Pure greedy improvement stops visiting unexplored actions — their Q estimates remain at the initial value and may never be corrected. Epsilon-soft policies maintain a minimum probability $\epsilon / |A|$ for every action, ensuring all state-action pairs are visited infinitely often.

## 3. GridWorld Environment for Control

We use the 5x5 GridWorld from Notebook 01 (Module 0), extended slightly. The agent must find a path from Start to Goal while avoiding holes. Since MC control learns without a model, it must discover the grid structure purely from interaction.

In [ ]:
# Purpose: Define the 5x5 GridWorld environment for MC control
# Key Concept: MC control interacts with the environment as a black box — no P or R needed

# Grid: 0=empty, 1=hole, 2=goal, 3=start
GRID = np.array([
    [3, 0, 0, 0, 0],
    [0, 1, 0, 1, 0],
    [0, 0, 0, 0, 0],
    [0, 1, 0, 1, 0],
    [0, 0, 0, 0, 2]
])

N_ROWS, N_COLS = GRID.shape
N_STATES  = N_ROWS * N_COLS  # 25
N_ACTIONS = 4

ACTION_UP, ACTION_DOWN, ACTION_LEFT, ACTION_RIGHT = 0, 1, 2, 3
ACTION_ARROWS  = {0: '↑', 1: '↓', 2: '←', 3: '→'}
CELL_REWARDS   = {0: -0.1, 1: -5.0, 2: 10.0, 3: -0.1}
DELTAS         = {0: (-1,0), 1:(1,0), 2:(0,-1), 3:(0,1)}

def pos_to_state(r, c):
    return r * N_COLS + c

def state_to_pos(s):
    return s // N_COLS, s % N_COLS

# Terminal states: holes and goal
TERMINAL_STATES = set()
for r in range(N_ROWS):
    for c in range(N_COLS):
        if GRID[r, c] in (1, 2):
            TERMINAL_STATES.add(pos_to_state(r, c))


class GridWorldEnv:
    """
    5x5 GridWorld: the agent interacts without knowing the transition model.
    Interface matches Blackjack from Notebook 1: reset() and step(action).
    """

    def __init__(self, grid):
        self.grid = grid
        start_pos = tuple(np.argwhere(grid == 3)[0])
        self.start_pos = start_pos
        self.agent_pos = start_pos
        self.done = False

    def reset(self):
        """Reset to start; return initial state index."""
        self.agent_pos = self.start_pos
        self.done = False
        return pos_to_state(*self.agent_pos)

    def step(self, action):
        """
        Execute action.

        Returns
        -------
        next_state : int
        reward : float
        done : bool
        """
        if self.done:
            raise RuntimeError("Call reset() first.")

        r, c = self.agent_pos
        dr, dc = DELTAS[action]
        nr, nc = r + dr, c + dc

        # Wall collision
        if not (0 <= nr < N_ROWS and 0 <= nc < N_COLS):
            nr, nc = r, c

        self.agent_pos = (nr, nc)
        cell_type = self.grid[nr, nc]
        reward = CELL_REWARDS[cell_type]
        self.done = cell_type in (1, 2)

        return pos_to_state(nr, nc), reward, self.done


env = GridWorldEnv(GRID)

# Print grid
print("5x5 GridWorld (S=start, H=hole, G=goal, .=empty):")
symbols = {0: '.', 1: 'H', 2: 'G', 3: 'S'}
for r in range(N_ROWS):
    print('  ' + '  '.join(symbols[GRID[r, c]] for c in range(N_COLS)))
print(f"\nTerminal states (flat): {sorted(TERMINAL_STATES)}")

## 4. Epsilon-Soft Policy Representation

An **epsilon-soft policy** assigns:
- Probability $1 - \epsilon + \epsilon / |A|$ to the greedy action
- Probability $\epsilon / |A|$ to each other action

When $\epsilon = 0$, this collapses to a deterministic greedy policy. When $\epsilon = 1$, it is uniform random. Values in between balance exploitation and exploration.

We represent the policy as a stochastic matrix $\pi \in \mathbb{R}^{|S| \times |A|}$ where $\pi[s, a]$ is the probability of choosing action $a$ in state $s$.

In [ ]:
# Purpose: Implement epsilon-soft policy creation and action sampling
# Key Concept: Epsilon-soft guarantees exploration while still exploiting the best-known action

def make_epsilon_soft_policy(Q, epsilon, n_actions):
    """
    Create an epsilon-soft policy from Q-table.

    Parameters
    ----------
    Q : np.ndarray (n_states, n_actions)
    epsilon : float  — exploration probability in [0, 1]
    n_actions : int

    Returns
    -------
    policy : np.ndarray (n_states, n_actions)
        pi[s, a] = epsilon/|A| for non-greedy actions
        pi[s, a*] = 1 - epsilon + epsilon/|A| for greedy action a*
    """
    n_states = Q.shape[0]
    policy   = np.ones((n_states, n_actions)) * epsilon / n_actions

    # Identify greedy action per state (random tie-breaking)
    greedy_actions = np.argmax(Q, axis=1)

    for s in range(n_states):
        policy[s, greedy_actions[s]] += 1.0 - epsilon

    # Sanity check
    assert np.allclose(policy.sum(axis=1), 1.0), "Policy rows must sum to 1."
    return policy


def sample_action(policy_row):
    """
    Sample an action from a probability vector.

    Parameters
    ----------
    policy_row : np.ndarray (n_actions,)

    Returns
    -------
    action : int
    """
    return np.random.choice(len(policy_row), p=policy_row)


# Quick demonstration
Q_demo   = np.zeros((N_STATES, N_ACTIONS))
Q_demo[0, ACTION_RIGHT] = 5.0  # Pretend RIGHT is best at state 0

pi_demo  = make_epsilon_soft_policy(Q_demo, epsilon=0.1, n_actions=N_ACTIONS)
print("Epsilon-soft policy at state 0 (RIGHT should be most likely):")
for a, prob in enumerate(pi_demo[0]):
    print(f"  {ACTION_ARROWS[a]}: {prob:.3f}")
print(f"Sum: {pi_demo[0].sum():.6f}")

## 5. On-Policy MC Control (First-Visit)

The algorithm:
```
Initialize Q(s,a) = 0, policy = uniform
Repeat:
    Generate episode under current policy
    For each (s, a) first visited in episode:
        Append G_t to Returns(s, a)
        Q(s, a) = average(Returns(s, a))
    Update policy: epsilon-soft greedy w.r.t. Q
```

The critical observation: we update Q and the policy after each complete episode. The policy used to generate the episode is the same policy being improved — this is **on-policy** learning.

In [ ]:
# Purpose: Implement on-policy MC control
# Key Concept: Q is updated from sampled returns; policy is immediately re-derived from Q

def generate_episode_from_policy(env, policy, max_steps=100):
    """
    Sample one episode using a stochastic policy matrix.

    Parameters
    ----------
    env : GridWorldEnv
    policy : np.ndarray (n_states, n_actions)
    max_steps : int

    Returns
    -------
    trajectory : list of (state, action, reward)
    """
    state = env.reset()
    trajectory = []

    for _ in range(max_steps):
        action = sample_action(policy[state])
        next_state, reward, done = env.step(action)
        trajectory.append((state, action, reward))
        state = next_state
        if done:
            break

    return trajectory


def mc_control_onpolicy(
    env, n_episodes, epsilon=0.1, gamma=0.99,
    n_states=None, n_actions=None
):
    """
    On-policy first-visit MC control with epsilon-soft policies.

    Parameters
    ----------
    env : GridWorldEnv
    n_episodes : int
    epsilon : float
    gamma : float
    n_states : int  (inferred from env if None)
    n_actions : int

    Returns
    -------
    Q : np.ndarray (n_states, n_actions)
    policy : np.ndarray (n_states, n_actions)
    episode_returns : list of float
    Q_history : list of np.ndarray  — snapshots every 1000 episodes
    """
    if n_states is None:
        n_states = N_STATES
    if n_actions is None:
        n_actions = N_ACTIONS

    # Initialize Q, return counters, and policy
    Q             = np.zeros((n_states, n_actions))
    returns_sum   = np.zeros((n_states, n_actions))
    returns_count = np.zeros((n_states, n_actions), dtype=int)
    policy        = np.ones((n_states, n_actions)) / n_actions  # uniform start

    episode_returns = []
    Q_history       = [Q.copy()]

    for ep_idx in range(n_episodes):
        # Step 1: Generate one episode under current epsilon-soft policy
        trajectory = generate_episode_from_policy(env, policy)

        # Step 2: Compute returns for each timestep
        rewards = [r for _, _, r in trajectory]
        T = len(rewards)
        G = 0.0
        returns = [0.0] * T
        for t in range(T - 1, -1, -1):
            G = rewards[t] + gamma * G
            returns[t] = G

        episode_returns.append(returns[0])  # G_0 = total discounted return

        # Step 3: Update Q using first-visit returns
        visited = set()
        for t, (state, action, _) in enumerate(trajectory):
            sa_pair = (state, action)
            if sa_pair not in visited:
                visited.add(sa_pair)
                returns_sum[state, action]   += returns[t]
                returns_count[state, action] += 1
                Q[state, action] = returns_sum[state, action] / returns_count[state, action]

        # Step 4: Improve policy (epsilon-soft greedy w.r.t. updated Q)
        policy = make_epsilon_soft_policy(Q, epsilon, n_actions)

        # Snapshot Q every 1000 episodes
        if (ep_idx + 1) % 1000 == 0:
            Q_history.append(Q.copy())

    return Q, policy, episode_returns, Q_history


N_EPISODES = 20_000
EPSILON    = 0.1
GAMMA      = 0.99

print(f"Running on-policy MC control ({N_EPISODES:,} episodes, epsilon={EPSILON})...")
Q_learned, policy_learned, returns_history, Q_snapshots = mc_control_onpolicy(
    env, n_episodes=N_EPISODES, epsilon=EPSILON, gamma=GAMMA
)

# Extract greedy policy
greedy_actions = np.argmax(Q_learned, axis=1)
print(f"\nTraining complete. Final episode return (last 500 avg): "
      f"{np.mean(returns_history[-500:]):.3f}")

## 6. Visualizing the Learned Policy and Q Function

We plot three things:
1. **Learning curve**: episode returns (smoothed) over training
2. **Learned policy**: arrow grid showing the greedy action in each state
3. **Max Q heatmap**: $\max_a Q(s,a)$ for each state — the agent's best-case value estimate

In [ ]:
# Purpose: Visualize learning curve, learned policy, and Q-value heatmap
# Key Concept: Arrow grids make the policy human-readable; Q heatmaps show value landscape

def smooth(values, window=200):
    """Rolling mean smoothing."""
    if len(values) < window:
        return np.array(values)
    return np.convolve(values, np.ones(window) / window, mode='valid')


def plot_gridworld_policy(Q, greedy_acts, grid, terminal_states, n_rows, n_cols,
                          ax, title):
    """
    Heatmap of max_a Q(s,a) with policy arrows overlaid.

    Parameters
    ----------
    Q : np.ndarray (n_states, n_actions)
    greedy_acts : np.ndarray (n_states,)
    grid : np.ndarray (n_rows, n_cols)
    terminal_states : set
    n_rows, n_cols : int
    ax : matplotlib Axes
    title : str
    """
    V_max  = np.max(Q, axis=1).reshape(n_rows, n_cols)
    cell_sym = {0: '', 1: 'H', 2: 'G', 3: 'S'}

    vmin = V_max.min()
    vmax = V_max.max()
    vcenter = 0.0 if vmin < 0 < vmax else (vmin + vmax) / 2
    norm = TwoSlopeNorm(vmin=vmin, vcenter=vcenter, vmax=max(vmax, vcenter + 0.01))

    im = ax.imshow(V_max, cmap='RdYlGn', norm=norm, interpolation='nearest')

    for r in range(n_rows):
        for c in range(n_cols):
            s = r * n_cols + c
            cell_type = grid[r, c]

            # Q value label
            ax.text(c, r - 0.25, f'{V_max[r, c]:.1f}', ha='center', va='center',
                    fontsize=7, color='black')

            # Arrow or terminal label
            if s in terminal_states:
                label = cell_sym[cell_type]
                ax.text(c, r + 0.2, label, ha='center', va='center',
                        fontsize=14, color='white', fontweight='bold')
            else:
                arrow = ACTION_ARROWS[greedy_acts[s]]
                ax.text(c, r + 0.2, arrow, ha='center', va='center',
                        fontsize=16, color='#1c7ed6', fontweight='bold')

    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_xticks([])
    ax.set_yticks([])
    return im


fig, axes = plt.subplots(1, 3, figsize=(17, 5))

# --- Plot 1: Learning curve ---
ax = axes[0]
smoothed = smooth(returns_history, window=200)
ax.plot(returns_history, color='#74c0fc', linewidth=0.5, alpha=0.4, label='Raw episode return')
ax.plot(np.arange(len(smoothed)) + 200, smoothed, color='#1c7ed6',
        linewidth=2.5, label=f'Smoothed (window=200)')
ax.set_xlabel('Episode', fontsize=11)
ax.set_ylabel('Return $G_0$', fontsize=11)
ax.set_title(f'MC Control Learning Curve\n($\\epsilon={EPSILON}$, $\\gamma={GAMMA}$)',
             fontsize=11, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
ax.axhline(0, color='black', linewidth=0.8, linestyle='--')

# --- Plot 2: Learned policy ---
im = plot_gridworld_policy(
    Q_learned, greedy_actions, GRID, TERMINAL_STATES, N_ROWS, N_COLS,
    axes[1], f'Learned Policy\n(max_a Q(s,a), arrows = greedy)'
)
plt.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04, label='max Q(s,a)')

# --- Plot 3: Per-action Q for a single state ---
ax = axes[2]
start_state = pos_to_state(*np.argwhere(GRID == 3)[0])
q_at_start  = Q_learned[start_state]
colors_bar  = ['#2f9e44', '#1c7ed6', '#e64980', '#f76707']
bars = ax.bar(range(N_ACTIONS), q_at_start, color=colors_bar, alpha=0.8, edgecolor='black')
ax.set_xticks(range(N_ACTIONS))
ax.set_xticklabels([f'{ACTION_ARROWS[a]}\n({["UP","DOWN","LEFT","RIGHT"][a]})'
                    for a in range(N_ACTIONS)], fontsize=10)
ax.set_ylabel('Q(start, a)', fontsize=11)
ax.set_title(f'Q values at Start State (state {start_state})', fontsize=11, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')
ax.axhline(0, color='black', linewidth=0.8)
for bar, val in zip(bars, q_at_start):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
            f'{val:.2f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig('mc_control_results.png', dpi=120, bbox_inches='tight')
plt.show()
print("Figure saved to mc_control_results.png")

## 7. Exploration-Exploitation Trade-Off: Varying Epsilon

The epsilon parameter directly controls how much the agent explores:
- **High epsilon**: lots of exploration, slow exploitation — the agent rarely commits to the best-known action
- **Low epsilon**: fast exploitation, little exploration — may get stuck in a locally good but globally suboptimal policy
- **Optimal epsilon**: depends on the problem; empirically we compare learning curves

We train agents with three epsilon values and compare their learning curves.

In [ ]:
# Purpose: Compare learning curves across epsilon values
# Key Concept: epsilon is a hyperparameter that trades off exploration and exploitation speed

EPSILONS = [0.01, 0.1, 0.5]
COLORS   = ['#2f9e44', '#1c7ed6', '#e64980']
eps_results = {}

for eps in EPSILONS:
    print(f"Training with epsilon={eps}...", end=' ', flush=True)
    np.random.seed(42)  # Same seed for fair comparison
    env_eps = GridWorldEnv(GRID)
    Q_eps, pi_eps, returns_eps, _ = mc_control_onpolicy(
        env_eps, n_episodes=N_EPISODES, epsilon=eps, gamma=GAMMA
    )
    eps_results[eps] = {
        'returns': returns_eps,
        'Q':       Q_eps,
        'pi':      pi_eps,
        'mean_last_500': np.mean(returns_eps[-500:])
    }
    print(f"done. Mean return (last 500): {eps_results[eps]['mean_last_500']:.3f}")

# Plot learning curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Left: Learning curves ---
ax = axes[0]
for eps, color in zip(EPSILONS, COLORS):
    smoothed = smooth(eps_results[eps]['returns'], window=300)
    ax.plot(np.arange(len(smoothed)) + 300, smoothed,
            color=color, linewidth=2.5, label=f'$\\epsilon={eps}$')

ax.set_xlabel('Episode', fontsize=12)
ax.set_ylabel('Return (smoothed)', fontsize=12)
ax.set_title('MC Control: Exploration-Exploitation Trade-off', fontsize=12, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.axhline(0, color='black', linewidth=0.8, linestyle='--')

# --- Right: Bar chart of final performance ---
ax = axes[1]
final_means = [eps_results[eps]['mean_last_500'] for eps in EPSILONS]
bars = ax.bar([str(eps) for eps in EPSILONS], final_means, color=COLORS, alpha=0.85,
               edgecolor='black')

for bar, val in zip(bars, final_means):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + (0.1 if val >= 0 else -0.5),
            f'{val:.3f}', ha='center', va='bottom', fontsize=11, fontweight='bold')

ax.set_xlabel('Epsilon', fontsize=12)
ax.set_ylabel('Mean return (last 500 episodes)', fontsize=12)
ax.set_title('Final Performance vs. Epsilon', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')
ax.axhline(0, color='black', linewidth=0.8)

plt.tight_layout()
plt.savefig('epsilon_comparison.png', dpi=120, bbox_inches='tight')
plt.show()
print("Figure saved to epsilon_comparison.png")

## 8. Exercises

### Exercise 2.1: Decaying Epsilon

**Task:** A common improvement is to decay epsilon over time: start with high exploration and gradually reduce it. Implement `mc_control_decaying_epsilon` where `epsilon_start=0.5`, `epsilon_end=0.01`, and epsilon decays linearly over the first 80% of episodes. Run it for 20,000 episodes and compare its final mean return (last 500 episodes) to the fixed-epsilon results above.

**Expected Output:** Final mean return of the decaying-epsilon agent, printed alongside fixed-epsilon baselines.

**Hints:**
<details>
<summary>Hint 1</summary>
At episode `t`, compute `epsilon = max(epsilon_end, epsilon_start - (epsilon_start - epsilon_end) * t / (0.8 * n_episodes))`.
</details>

<details>
<summary>Hint 2 (more specific)</summary>
Copy the `mc_control_onpolicy` function and add two lines at the start of the episode loop:
```python
decay_progress = min(ep_idx / (0.8 * n_episodes), 1.0)
epsilon = epsilon_start + (epsilon_end - epsilon_start) * decay_progress
```
Then use `epsilon` when calling `make_epsilon_soft_policy`.
</details>

In [ ]:
# YOUR CODE HERE
# ---------------
def mc_control_decaying_epsilon(
    env, n_episodes, epsilon_start=0.5, epsilon_end=0.01, gamma=0.99
):
    """
    On-policy MC control with linearly decaying epsilon.

    Epsilon decays from epsilon_start to epsilon_end over the first 80% of episodes,
    then stays at epsilon_end.

    Returns
    -------
    Q : np.ndarray (n_states, n_actions)
    policy : np.ndarray (n_states, n_actions)
    episode_returns : list of float
    epsilon_trace : list of float  — epsilon value used at each episode
    """
    # YOUR IMPLEMENTATION HERE
    Q              = None
    policy         = None
    episode_returns = []
    epsilon_trace  = []
    return Q, policy, episode_returns, epsilon_trace


np.random.seed(42)
env_decay = GridWorldEnv(GRID)
Q_decay, pi_decay, returns_decay, eps_trace = mc_control_decaying_epsilon(
    env_decay, n_episodes=N_EPISODES, epsilon_start=0.5, epsilon_end=0.01, gamma=GAMMA
)

mean_return_decay = None  # Mean return of last 500 episodes

In [ ]:
# AUTO-GRADED TESTS - Do not modify
# ----------------------------------
def test_exercise_2_1():
    assert Q_decay is not None, "Return Q from mc_control_decaying_epsilon!"
    assert Q_decay.shape == (N_STATES, N_ACTIONS), \
        f"Q_decay should have shape ({N_STATES}, {N_ACTIONS})"

    assert len(returns_decay) == N_EPISODES, \
        f"returns_decay should have {N_EPISODES} entries, got {len(returns_decay)}"

    assert len(eps_trace) == N_EPISODES, \
        f"epsilon_trace should have {N_EPISODES} entries"

    # Epsilon should decay: first value > last value
    assert eps_trace[0] > eps_trace[-1], \
        f"Epsilon should decay: eps_trace[0]={eps_trace[0]:.3f} should > eps_trace[-1]={eps_trace[-1]:.3f}"

    # Epsilon at end should be close to epsilon_end
    assert abs(eps_trace[-1] - 0.01) < 0.005, \
        f"Final epsilon should be ~0.01, got {eps_trace[-1]:.4f}"

    assert mean_return_decay is not None, "Compute mean_return_decay (last 500 episodes)!"

    print(f"Exercise 2.1 passed!")
    print(f"  Decaying epsilon: {eps_trace[0]:.3f} -> {eps_trace[-1]:.3f}")
    print(f"  Mean return (last 500): {mean_return_decay:.4f}")
    print(f"  Compare to fixed epsilon:")
    for eps in EPSILONS:
        print(f"    eps={eps}: {eps_results[eps]['mean_last_500']:.4f}")

test_exercise_2_1()

### Exercise 2.2: Monte Carlo vs. Greedy — Policy Quality Comparison

**Task:** Take the Q function learned by MC control (with epsilon=0.1) and extract the fully greedy policy (epsilon=0). Evaluate this greedy policy by running 1,000 test episodes (no exploration) and computing the fraction that reach the goal (reward = +10 within the episode). Compare this to 1,000 episodes with a purely random policy. Report goal-reaching rates for both.

**Expected Output:** Two percentages: % episodes reaching the goal for MC-learned vs. random policy.

**Hints:**
<details>
<summary>Hint 1</summary>
The greedy policy is `np.argmax(Q_learned, axis=1)`. To execute it deterministically, select `action = greedy_policy[state]` without random sampling.
</details>

<details>
<summary>Hint 2 (more specific)</summary>
An episode reaches the goal if any step produces reward = +10.0. Track this per episode and divide by total episodes to get the rate.
</details>

In [ ]:
# YOUR CODE HERE
# ---------------
N_TEST = 1_000

def evaluate_greedy(Q, env, n_test=1000, max_steps=100):
    """
    Evaluate the greedy policy from Q on n_test episodes.

    Returns
    -------
    goal_rate : float  — fraction of episodes where goal (reward=10) was reached
    mean_return : float
    """
    # YOUR IMPLEMENTATION HERE
    goal_rate   = None
    mean_return = None
    return goal_rate, mean_return


def evaluate_random(env, n_test=1000, max_steps=100):
    """
    Evaluate a uniform random policy on n_test episodes.

    Returns
    -------
    goal_rate : float
    mean_return : float
    """
    # YOUR IMPLEMENTATION HERE
    goal_rate   = None
    mean_return = None
    return goal_rate, mean_return


np.random.seed(0)
env_test = GridWorldEnv(GRID)

goal_rate_mc,  mean_return_mc  = evaluate_greedy(Q_learned, env_test, N_TEST)
goal_rate_rand, mean_return_rand = evaluate_random(env_test, N_TEST)

In [ ]:
# AUTO-GRADED TESTS - Do not modify
# ----------------------------------
def test_exercise_2_2():
    assert goal_rate_mc   is not None, "Compute goal_rate_mc from evaluate_greedy!"
    assert goal_rate_rand is not None, "Compute goal_rate_rand from evaluate_random!"
    assert mean_return_mc   is not None, "Compute mean_return_mc!"
    assert mean_return_rand is not None, "Compute mean_return_rand!"

    # Rates should be in [0, 1]
    assert 0.0 <= goal_rate_mc   <= 1.0, f"goal_rate_mc must be in [0,1], got {goal_rate_mc}"
    assert 0.0 <= goal_rate_rand <= 1.0, f"goal_rate_rand must be in [0,1], got {goal_rate_rand}"

    # MC-learned greedy policy should reach the goal more often than random
    assert goal_rate_mc >= goal_rate_rand, (
        f"MC-learned policy ({goal_rate_mc:.1%}) should reach goal more often than random "
        f"({goal_rate_rand:.1%}). Check your Q_learned or greedy policy extraction."
    )

    print(f"Exercise 2.2 passed!")
    print(f"  MC-learned greedy policy: {goal_rate_mc:.1%} goal rate, "
          f"mean return = {mean_return_mc:.3f}")
    print(f"  Random policy:            {goal_rate_rand:.1%} goal rate, "
          f"mean return = {mean_return_rand:.3f}")
    improvement = (goal_rate_mc - goal_rate_rand) / max(goal_rate_rand, 0.001)
    print(f"  Improvement over random: {improvement:.1%} relative increase in goal rate")

test_exercise_2_2()

## Key Takeaways

1. **MC control = MC prediction + policy improvement**, repeated. No model is needed: Q is estimated from sampled returns, and the policy is improved greedily from Q.
2. **Epsilon-soft policies ensure exploration**: every (state, action) pair must be visited infinitely often for Q estimates to converge. The minimum probability $\epsilon/|A|$ per action guarantees this.
3. **Epsilon is a hyperparameter** that trades off exploration speed against exploitation efficiency. A well-tuned decaying schedule typically outperforms any fixed epsilon.
4. **On-policy MC** evaluates and improves the same policy that generates the data. The policy is always epsilon-soft, so the optimal policy found is the best possible epsilon-soft policy — not the globally optimal deterministic policy.
5. **The learning curve** reveals training stability: a noisy curve with high variance suggests insufficient exploration or too long episodes; a smooth curve indicates good balance.
6. **Arrow grids** are the standard policy visualization for 2D gridworlds and make it immediately obvious whether the agent is navigating sensibly.

## What's Next

Module 3 introduces **Temporal Difference (TD) methods** — the best of both worlds. TD methods are model-free like MC, but they update values after every step (not just at episode end), like DP. This makes them applicable to continuing (non-episodic) tasks and significantly faster to converge. SARSA and Q-learning are the canonical TD control algorithms.